In [1]:
from pyiceberg.catalog import load_catalog
import duckdb

PROJECT = "/Users/Shared/academy/learner/projects/ethereum_iceberg"
catalog = load_catalog(
    "local",
    type="sql",
    uri=f"sqlite:///{PROJECT}/catalog/pyiceberg_catalog.db",
    warehouse=f"file://{PROJECT}/warehouse",
)
S3 = "s3://aws-public-blockchain/v1.0/eth"
con = duckdb.connect()
con.execute("set s3_region = 'us-east-2'")
catalog.list_namespaces()

[('ref',), ('raw',)]

In [2]:
from pyiceberg.expressions import EqualTo
blocks = catalog.load_table("raw.blocks")
blocks.scan(row_filter="transaction_count = 0").to_arrow().num_rows

127

In [3]:
blocks.delete(delete_filter="transaction_count = 0")
blocks.current_snapshot().summary.additional_properties

{'added-files-size': '129720010',
 'removed-files-size': '129901921',
 'added-data-files': '4',
 'deleted-data-files': '4',
 'added-records': '659769',
 'deleted-records': '659896',
 'total-data-files': '5',
 'total-delete-files': '0',
 'total-records': '666945',
 'total-files-size': '131174933',
 'total-position-deletes': '0',
 'total-equality-deletes': '0'}

In [4]:
import datetime
nov1 = con.execute(f"select * exclude (logs_bloom, last_modified) from read_parquet('{S3}/blocks/date=2026-11-01/*.parquet') where transaction_count > 0").to_arrow_table()
blocks.overwrite(nov1, overwrite_filter=EqualTo("date", datetime.date(2026, 11, 1)))
blocks.current_snapshot().summary.additional_properties

{'added-files-size': '1454923',
 'added-data-files': '1',
 'added-records': '7176',
 'total-data-files': '5',
 'total-delete-files': '0',
 'total-records': '666945',
 'total-files-size': '131174933',
 'total-position-deletes': '0',
 'total-equality-deletes': '0'}

In [5]:
nov2 = con.execute(f"select * exclude (logs_bloom, last_modified) from read_parquet('{S3}/blocks/date=2026-11-0[12]/*.parquet') where transaction_count > 0").to_arrow_table()
result = blocks.upsert(nov2, join_cols=["number"])
result.rows_updated, result.rows_inserted

(0, 7176)